# EcoVision — Week 3: TensorFlow Lite Export & Parity

**Goal:** Convert the preselected frozen MobileNetV2 epoch-8 checkpoint to a **float32** TensorFlow Lite artifact, verify the input/output contract, and compare TensorFlow and TFLite on identical RGB preprocessed images. No retraining, tuning or new test-set evaluation.

**Execution:** Run cells **A → D** in order, using kernel `Python (EcoVision ML - WSL GPU)`. Use the validation split for parity so that the final test set remains a one-time evaluation. Stop at any failed assertion and investigate; do not change model parameters based on test performance.

The mobile application must still implement JPEG decode, RGB conversion and resizing (224×224, bilinear + antialias matching the training pipeline); `Rescaling(1/127.5, offset=-1)` is already **inside** the model. TensorFlow/TFLite parity alone does not establish Flutter preprocessing parity or Android latency.

## A. Environment and model contract preflight

Verify the exact `.keras` checkpoint SHA-256, its saved label mapping, and the expected model input/output. Do **not** touch the held-out test set.

In [ ]:
from pathlib import Path
import hashlib
import json
import platform

import numpy as np
import pandas as pd
import tensorflow as tf

EXPECTED_CHECKPOINT_SHA256 = "0d560451fbd667f42c852752436a0ffc77f8068b8bd3b326ded4bd16bc3e6023"
EXPECTED_LABELS = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]


def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "ml/notebooks/02_mobilenetv2_baseline.ipynb").is_file() and (candidate / "ml/data/splits/v1/val.csv").is_file():
            return candidate.resolve()
    raise FileNotFoundError("Cannot find ecovision-flutter-ml root from current Jupyter working directory")


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


ROOT = find_project_root()
CHECKPOINT = ROOT / "ml/reports/generated/week2/mobilenetv2_frozen_20261103_155149/best_model.keras"
LABELS_PATH = ROOT / "ml/models/v1/labels.json"
MODEL_DIR = ROOT / "ml/models/v1"
MODEL_PATH = MODEL_DIR / "ecovision_mobilenetv2_float32_v1.tflite"
METADATA_PATH = MODEL_DIR / "model_metadata.json"
VAL_CSV = ROOT / "ml/data/splits/v1/val.csv"
DATA_ROOT = ROOT / "ml/data/raw/dataset-resized"
REPORT_DIR = ROOT / "ml/reports/generated/week3"

assert CHECKPOINT.is_file(), CHECKPOINT
assert LABELS_PATH.is_file(), LABELS_PATH
assert VAL_CSV.is_file(), VAL_CSV
assert DATA_ROOT.is_dir(), DATA_ROOT
assert sha256_file(CHECKPOINT) == EXPECTED_CHECKPOINT_SHA256, "Week 2 checkpoint hash changed"
labels = json.loads(LABELS_PATH.read_text(encoding="utf-8"))
assert labels == EXPECTED_LABELS, "Unexpected labels.json order"

model = tf.keras.models.load_model(CHECKPOINT, compile=False)
assert tuple(model.input_shape[1:]) == (224, 224, 3)
assert tuple(model.output_shape[1:]) == (6,)
assert model.inputs[0].dtype == "float32"
assert model.outputs[0].dtype == "float32"
rescaling = [layer for layer in model.layers if isinstance(layer, tf.keras.layers.Rescaling)]
assert len(rescaling) == 1, "Expected exactly one internal rescaling layer"
assert np.isclose(float(rescaling[0].scale), 1 / 127.5)
assert np.isclose(float(rescaling[0].offset), -1.0)

print("PASS: frozen epoch-8 checkpoint and SHA-256")
print("PASS: labels and model contract")
print("TensorFlow:", tf.__version__)
print("Python:", platform.python_version())
print("Input:", model.input_shape, model.inputs[0].dtype)
print("Output:", model.output_shape, model.outputs[0].dtype)
print("Target file:", MODEL_PATH)

## B. Convert to **float32** TFLite and inspect exported tensor contract

Use TensorFlow's built-in converter with **no quantization** or optimization. This produces a float32 baseline for Flutter integration and numeric parity checks. If the target file already exists, the notebook stops instead of silently overwriting a versioned artifact.

In [ ]:
assert not MODEL_PATH.exists(), (
    f"Versioned model already exists: {MODEL_PATH}. "
    "Do not overwrite silently. If export completed earlier, skip this cell and continue to C."
)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
# No converter.optimizations / quantization: keep float32 baseline.
tflite_bytes = converter.convert()
assert len(tflite_bytes) > 0, "Conversion returned an empty model"

probe = tf.lite.Interpreter(model_content=tflite_bytes)
probe.allocate_tensors()
inputs = probe.get_input_details()
outputs = probe.get_output_details()
assert len(inputs) == 1, f"Expected one input, got {len(inputs)}"
assert len(outputs) == 1, f"Expected one output, got {len(outputs)}"
assert tuple(inputs[0]["shape"]) == (1, 224, 224, 3), inputs[0]["shape"]
assert tuple(outputs[0]["shape"]) == (1, 6), outputs[0]["shape"]
assert inputs[0]["dtype"] == np.float32, inputs[0]["dtype"]
assert outputs[0]["dtype"] == np.float32, outputs[0]["dtype"]

MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH.write_bytes(tflite_bytes)
print("PASS: converted Keras checkpoint to float32 TensorFlow Lite")
print("Input shape/dtype:", inputs[0]["shape"].tolist(), inputs[0]["dtype"])
print("Input shape_signature:", inputs[0]["shape_signature"].tolist())
print("Output shape/dtype:", outputs[0]["shape"].tolist(), outputs[0]["dtype"])
print("Model size (MiB):", round(MODEL_PATH.stat().st_size / (1024 ** 2), 3))
print("TFLite SHA-256:", sha256_file(MODEL_PATH))

## C. TensorFlow vs TFLite parity on **validation images**

Use **2 deterministic images per class (12 total)** from `val.csv`. Image bytes are checked against SHA-256 manifests. Both runtimes receive the *same* `[1,224,224,3]` RGB float32 tensor in range 0–255; model-internal rescaling is applied by both. Compare all six probability outputs and top-1 labels. This is a parity smoke test, not a new accuracy benchmark.

In [ ]:

# ============================================================
# C. TensorFlow vs TFLite CPU Parity Test
# EcoVision Week 3 — MobileNetV2 Float32 v1
# ============================================================

# 1. Verify converted model and validation manifest
assert MODEL_PATH.is_file(), (
    f"Missing converted model: {MODEL_PATH}; run cell B first"
)

val = pd.read_csv(VAL_CSV, dtype=str)

assert {"path", "label", "sha256", "split"}.issubset(val.columns)
assert val["split"].eq("val").all()
assert set(val["label"]) == set(labels)

# Select 2 validation images from each of the 6 classes
samples = (
    val.groupby("label", sort=False)
    .head(2)
    .reset_index(drop=True)
)

assert len(samples) == 12, (
    f"Expected 12 balanced validation images, got {len(samples)}"
)
assert samples["label"].value_counts().eq(2).all()


# 2. Image preprocessing — identical to Week 2
def load_rgb_224(relative_path):
    path = DATA_ROOT / relative_path
    assert path.is_file(), path

    image_bytes = tf.io.read_file(str(path))

    image = tf.io.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        (224, 224),
        method="bilinear",
        antialias=True
    )

    image = tf.ensure_shape(image, [224, 224, 3])

    return tf.expand_dims(
        tf.cast(image, tf.float32), 0
    ).numpy()


# 3. Initialize TFLite interpreter (CPU)
interpreter = tf.lite.Interpreter(
    model_path=str(MODEL_PATH)
)
interpreter.allocate_tensors()

input_detail = interpreter.get_input_details()[0]
output_detail = interpreter.get_output_details()[0]

assert tuple(input_detail["shape"]) == (1, 224, 224, 3)
assert tuple(output_detail["shape"]) == (1, 6)
assert input_detail["dtype"] == np.float32
assert output_detail["dtype"] == np.float32


# 4. Run Keras CPU and TFLite CPU inference
records = []
keras_outputs = []
lite_outputs = []

for row in samples.itertuples(index=False):
    image_path = DATA_ROOT / row.path

    assert sha256_file(image_path) == row.sha256, (
        f"Validation image changed: {row.path}"
    )

    x = load_rgb_224(row.path)

    # FIX: Explicitly execute Keras inference on CPU
    with tf.device("/CPU:0"):
        x_cpu = tf.convert_to_tensor(x)
        keras_proba = model(
            x_cpu,
            training=False
        ).numpy()[0]

    # TFLite inference on CPU
    interpreter.set_tensor(input_detail["index"], x)
    interpreter.invoke()

    lite_proba = interpreter.get_tensor(
        output_detail["index"]
    )[0]

    keras_outputs.append(keras_proba)
    lite_outputs.append(lite_proba)

    records.append({
        "path": row.path,
        "true_label": row.label,
        "tf_predicted_label": labels[
            int(np.argmax(keras_proba))
        ],
        "tflite_predicted_label": labels[
            int(np.argmax(lite_proba))
        ],
        "max_abs_diff": float(
            np.max(np.abs(keras_proba - lite_proba))
        ),
    })


# 5. Validate prediction arrays
keras_outputs = np.asarray(keras_outputs)
lite_outputs = np.asarray(lite_outputs)

assert keras_outputs.shape == lite_outputs.shape == (12, 6)

assert (
    np.isfinite(keras_outputs).all()
    and np.isfinite(lite_outputs).all()
)

assert np.allclose(
    keras_outputs.sum(axis=1),
    1.0,
    atol=1e-4
)

assert np.allclose(
    lite_outputs.sum(axis=1),
    1.0,
    atol=1e-4
)


# 6. Calculate parity metrics
max_abs_diff = float(
    np.max(np.abs(keras_outputs - lite_outputs))
)

mean_abs_diff = float(
    np.mean(np.abs(keras_outputs - lite_outputs))
)

top1_match_count = int(
    np.count_nonzero(
        np.argmax(keras_outputs, axis=1)
        ==
        np.argmax(lite_outputs, axis=1)
    )
)

# Keep the original tolerance — do not loosen it
PARITY_ATOL = 1e-3
PARITY_RTOL = 1e-3

numeric_close = bool(
    np.allclose(
        keras_outputs,
        lite_outputs,
        atol=PARITY_ATOL,
        rtol=PARITY_RTOL
    )
)


# 7. Display results
print("=== ECOVISION WEEK 3 — CPU PARITY TEST ===")
print("Backend: Keras CPU vs TFLite CPU")
print("Validation sample count:", len(samples))
print(
    "Top-1 matches:",
    top1_match_count,
    "/",
    len(samples)
)
print(
    "Max absolute probability difference:",
    f"{max_abs_diff:.8f}"
)
print(
    "Mean absolute probability difference:",
    f"{mean_abs_diff:.8f}"
)
print(
    f"Numeric closeness "
    f"(atol={PARITY_ATOL}, rtol={PARITY_RTOL}):",
    numeric_close
)

display(pd.DataFrame(records))


# 8. Final parity assertions
assert top1_match_count == len(samples), (
    "TF/TFLite top-1 mismatch: inspect records"
)

assert numeric_close, (
    "TF/TFLite CPU numeric difference exceeds tolerance"
)

print("\nPASS: TF/TFLite CPU parity on 12 validation images")


In [ ]:

# Diagnostic: TF vs TFLite numerical differences
# Uses existing outputs from Cell C.

abs_diff = np.abs(keras_outputs - lite_outputs)

tolerance = (
    PARITY_ATOL
    + PARITY_RTOL * np.abs(lite_outputs)
)

failed = abs_diff > tolerance

diagnostics = pd.DataFrame({
    "image": samples["path"].tolist(),
    "tf_prediction": [
        labels[i] for i in np.argmax(keras_outputs, axis=1)
    ],
    "tflite_prediction": [
        labels[i] for i in np.argmax(lite_outputs, axis=1)
    ],
    "max_abs_diff": abs_diff.max(axis=1),
    "failed_probabilities": failed.sum(axis=1),
})

print("=== ECOVISION PARITY DIAGNOSTICS ===")
print("Top-1 matches:", top1_match_count, "/ 12")
print("Maximum absolute difference:", max_abs_diff)
print("Mean absolute difference:", mean_abs_diff)
print("Numeric closeness:", numeric_close)
print("Probabilities outside tolerance:", failed.sum(), "/ 72")

display(
    diagnostics.sort_values(
        "max_abs_diff",
        ascending=False
    )
)

worst_image, worst_class = np.unravel_index(
    np.argmax(abs_diff),
    abs_diff.shape
)

print("\n=== WORST DIFFERENCE ===")
print("Image:", samples.iloc[worst_image]["path"])
print("Class:", labels[worst_class])
print("TensorFlow:", keras_outputs[worst_image, worst_class])
print("TFLite:", lite_outputs[worst_image, worst_class])
print("Absolute difference:", abs_diff[worst_image, worst_class])
print("Allowed tolerance:", tolerance[worst_image, worst_class])


In [ ]:
print("=== FINAL PARITY VERIFICATION ===")
print("Top-1 matches:", top1_match_count, "/ 12")
print("Max absolute difference:", max_abs_diff)
print("Mean absolute difference:", mean_abs_diff)
print("Numeric closeness:", numeric_close)

assert top1_match_count == 12
assert numeric_close

print("\nPASS: EcoVision TFLite CPU parity verified!")

In [ ]:

# Week 3 - CPU vs TFLite parity diagnosis
# Do not modify the model or tolerance.

cpu_outputs = []

for path in samples["path"]:
    x = load_rgb_224(path)

    with tf.device("/CPU:0"):
        x_cpu = tf.convert_to_tensor(x)
        prediction = model(
            x_cpu,
            training=False
        ).numpy()[0]

    cpu_outputs.append(prediction)

cpu_outputs = np.asarray(cpu_outputs)

assert cpu_outputs.shape == (12, 6)
assert np.isfinite(cpu_outputs).all()

def compare_outputs(name, a, b):
    differences = np.abs(a - b)

    top1_matches = int(np.sum(
        np.argmax(a, axis=1) ==
        np.argmax(b, axis=1)
    ))

    close = np.isclose(
        a, b,
        atol=PARITY_ATOL,
        rtol=PARITY_RTOL
    )

    print("\n" + name)
    print("Top-1 matches:", top1_matches, "/ 12")
    print("Max absolute difference:", float(differences.max()))
    print("Mean absolute difference:", float(differences.mean()))
    print("Probabilities outside tolerance:", int((~close).sum()), "/ 72")
    print("Numeric closeness:", bool(close.all()))

compare_outputs(
    "Keras original vs TFLite",
    keras_outputs,
    lite_outputs
)

compare_outputs(
    "Keras CPU vs TFLite CPU",
    cpu_outputs,
    lite_outputs
)

compare_outputs(
    "Keras original vs Keras CPU",
    keras_outputs,
    cpu_outputs
)


## D. Save reproducible export and parity metadata

Only run after A–C pass. The `.tflite` artifact and `model_metadata.json` are versioned model-contract outputs; the detailed validation parity report goes into the git-ignored `ml/reports/generated/week3/`. Do not push raw dataset, secrets, or generated large reports.

In [ ]:
assert MODEL_PATH.is_file()
assert top1_match_count == 12 and numeric_close
assert labels == EXPECTED_LABELS
REPORT_DIR.mkdir(parents=True, exist_ok=True)

parity_report = {
    "model_version": "v1.0.0",
    "checkpoint_sha256": EXPECTED_CHECKPOINT_SHA256,
    "tflite_sha256": sha256_file(MODEL_PATH),
    "tensorflow_version": tf.__version__,
    "split": "validation_v1",
    "sampling": "first two rows per class in val.csv",
    "sample_count": 12,
    "top1_matches": top1_match_count,
    "max_abs_probability_difference": max_abs_diff,
    "mean_abs_probability_difference": mean_abs_diff,
    "atol": PARITY_ATOL,
    "rtol": PARITY_RTOL,
    "numeric_allclose": numeric_close,
    "rows": records,
}
(REPORT_DIR / "tflite_parity_validation.json").write_text(
    json.dumps(parity_report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)

metadata = {
    "model_version": "v1.0.0",
    "architecture": "MobileNetV2 frozen ImageNet backbone, custom six-class head",
    "source_checkpoint": "mobilenetv2_frozen_20261103_155149/best_model.keras",
    "source_checkpoint_sha256": EXPECTED_CHECKPOINT_SHA256,
    "tflite_filename": MODEL_PATH.name,
    "tflite_sha256": sha256_file(MODEL_PATH),
    "tflite_size_bytes": MODEL_PATH.stat().st_size,
    "conversion": "TFLite default float32; no quantization",
    "tensorflow_version": tf.__version__,
    "input": {
        "shape": [1, 224, 224, 3],
        "dtype": "float32",
        "color": "RGB",
        "pixel_value_range": "0 to 255",
        "external_preprocessing": "decode RGB, resize bilinear with antialias=True to 224x224",
        "internal_rescaling": "x / 127.5 - 1",
    },
    "output": {"shape": [1, 6], "dtype": "float32", "type": "softmax probabilities"},
    "label_order": labels,
    "parity": {
        "split": "validation_v1",
        "samples": 12,
        "top1_matches": top1_match_count,
        "max_abs_probability_difference": max_abs_diff,
        "atol": PARITY_ATOL,
        "rtol": PARITY_RTOL,
    },
    "limitations": [
        "TensorFlow/TFLite same-input parity does not prove Flutter image preprocessing parity",
        "Android real-device inference latency and offline integration are pending",
        "Real-world smartphone-photo evaluation is pending",
    ],
}
if METADATA_PATH.is_file():
    prior = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
    assert prior == metadata, "Existing v1 metadata differs: do not overwrite a versioned artifact"
else:
    METADATA_PATH.write_text(json.dumps(metadata, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

print("PASS: saved metadata and validation parity report")
print("TFLite model:", MODEL_PATH)
print("Model metadata:", METADATA_PATH)
print("Detailed parity:", REPORT_DIR / "tflite_parity_validation.json")
print("Model size (MiB):", round(MODEL_PATH.stat().st_size / (1024 ** 2), 3))
print("TFLite SHA256:", sha256_file(MODEL_PATH))

## Next steps (not claimed completed)

1. Review TensorFlow/TFLite parity values; investigate any mismatches instead of silently relaxing tolerances.
2. Record **observed** model size and parity results in a tracked `ml/reports/week3_model_evaluation.md` and model card.
3. Confirm `.gitignore` policy and keep locally generated images/reports excluded. Only intentionally stage versioned artifacts; model binaries are optional to commit depending on repository policy/GitHub Release strategy.
4. In Weeks 5–7, verify Flutter preprocessing parity using identical RGB tensors, offline Android inference, device latency, and separate real-world smartphone photos.

**Sources:** TensorFlow `tf.lite.TFLiteConverter.from_keras_model`, `tf.lite.Interpreter` official API; Week 2 / Week 3 EcoVision model contract and training pipeline.